# Data Search Agent Testing Notebook for SMEs

This notebook allows Subject Matter Experts (SMEs) to test and compare three different configurations of the Data Search Agent:

1. **Legacy Reranker (No LLM)** - Original ranking system without LLM-based reranking
2. **LLM Reranker (Default Config)** - LLM-based reranking with default CMR criteria
3. **LLM Reranker (Custom Config)** - LLM-based reranking with custom criteria and weights

## Overview

The Data Search Agent helps researchers discover scientific datasets by:
- Breaking down complex queries into searchable topics and decompositions
- Searching NASA's Common Metadata Repository (CMR) for relevant datasets
- Ranking results based on relevance, quality, and usability

### What's Different Between Configurations?

- **Legacy Reranker**: Uses the original LLM-based approach filtering and ranking components (slower, more LLM calls)
- **Default LLM Reranker**: Uses new efficient LLM reranker with CMR-specific criteria (faster, fewer LLM calls)
- **Custom LLM Reranker**: Allows you to define your own scoring criteria, weights, and categories

## Setup

Make sure your environment is properly configured with:
- API keys set in `.env` file
- CMR MCP server running (if testing locally)
- Required dependencies installed (`uv sync`)

In [16]:
# Import required modules
import json
from pathlib import Path
from datetime import datetime

from akd.agents.data_search import DataSearchAgent, DataSearchAgentConfig, DataSearchAgentInputSchema
from akd.agents.data_search.handlers import CMRHandlerConfig
from akd.tools.reranker import LLMRerankerToolConfig, ScoringCriterion, ScoringCategory

## Configuration

Set your test query and common parameters here:

In [ ]:
# Test query - change this to test different queries
TEST_QUERY = "What datasets are available on sea surface temperature anomalies in the Pacific Ocean over the last 20 years?"

# Common configuration
MODEL = "gpt-4o-mini"  # or "gpt-5-mini" for better quality
OUTPUT_DIR = "sme_testing_results"
CMR_ENDPOINT = "http://localhost:8080/mcp/cmr/mcp"  # CMR MCP endpoint
DEBUG = False  # Set to True for more verbose output

# Create output directory
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

print(f"Test Query: {TEST_QUERY}")
print(f"Model: {MODEL}")
print(f"CMR Endpoint: {CMR_ENDPOINT}")
print(f"Output Directory: {OUTPUT_DIR}")

Test Query: What datasets are available on sea surface temperature anomalies in the Pacific Ocean over the last 20 years?
Model: gpt-4o-mini
CMR Endpoint: http://localhost:8080/mcp/cmr/mcp
Output Directory: evaluations/sme_testing_results


---

## Test 1: Legacy Reranker (No LLM Reranker)

This configuration uses the original ranking system without the new LLM reranker. It uses:
- Per-approach collection filtering with LLMs
- Final cross-approach ranking with LLMs
- Multiple LLM calls for filtering and ranking

### Configuration Details:
- `use_llm_reranker=False` - Disables the new LLM reranker
- Uses legacy `approach_filtering_model` and `final_ranking_model`

In [18]:
print("\n" + "="*80)
print("TEST 1: LEGACY RERANKER (NO LLM RERANKER)")
print("="*80 + "\n")

# Configure CMR handler WITHOUT LLM reranker (matches run_single_query.py)
cmr_config_legacy = CMRHandlerConfig(
    use_llm_reranker=False,  # Use legacy ranking system
    approach_filtering_model=MODEL,
    final_ranking_model=MODEL,
    mcp_endpoint=CMR_ENDPOINT,  
)

# Configure agent
config_legacy = DataSearchAgentConfig(
    cmr=cmr_config_legacy,
    topic_splitting_model=MODEL,
    scientific_decomposition_model=MODEL,
    repository_routing_model=MODEL,
    auto_save=True,
    output_subdir=str(output_path / "legacy"),
    debug=DEBUG,
)

# Create agent
agent_legacy = DataSearchAgent(config=config_legacy, debug=DEBUG)

print("Configuration:")
print(f"  - use_llm_reranker: False")
print(f"  - approach_filtering_model: {MODEL}")
print(f"  - final_ranking_model: {MODEL}")
print(f"  - mcp_endpoint: {CMR_ENDPOINT}")
print("\nRunning search...")


TEST 1: LEGACY RERANKER (NO LLM RERANKER)

Configuration:
  - use_llm_reranker: False
  - approach_filtering_model: gpt-4o-mini
  - final_ranking_model: gpt-4o-mini
  - mcp_endpoint: http://localhost:8080/mcp/cmr/mcp

Running search...


In [ ]:
# Run the search
input_params_legacy = DataSearchAgentInputSchema(query=TEST_QUERY)
print("\nRunning search...")
result_legacy = await agent_legacy.arun(input_params_legacy)


# Display results
print("\n" + "="*80)
print("RESULTS - LEGACY RERANKER")
print("="*80)
print(f"Search ID: {result_legacy.search_metadata.get('search_id')}")
print(f"Topics Processed: {result_legacy.search_metadata.get('topics_processed', 0)}")
print(f"Total CMR Results: {result_legacy.total_cmr_results}")
print(f"Total Filtered Results: {result_legacy.total_filtered_results}")
print(f"Duration: {result_legacy.search_metadata.get('duration_seconds', 0):.1f}s")

# Save results
output_file_legacy = output_path / "legacy" / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
output_file_legacy.parent.mkdir(parents=True, exist_ok=True)
with open(output_file_legacy, "w") as f:
    json.dump(result_legacy.model_dump(), f, indent=2)



In [20]:
print(f"\nResults saved to: {output_file_legacy}")


Results saved to: evaluations/sme_testing_results/legacy/results_20251124_182242.json


In [21]:
# Display top 5 results
print("\nTop 5 Collections (Legacy Reranker):")
print("="*80)

if result_legacy.summary and result_legacy.summary.topics:
    for topic in result_legacy.summary.topics[:1]:  # Show first topic
        for decomp in topic.decompositions[:1]:  # Show first decomposition
            for idx, (concept_id, title) in enumerate(decomp.collections[:5], 1):
                print(f"{idx}. {concept_id}")
                print(f"   {title[:100]}...")
                print()


Top 5 Collections (Legacy Reranker):
1. C1996881146-POCLOUD
   GHRSST Level 4 MUR Global Foundation Sea Surface Temperature Analysis (v4.1)...

2. C2036880657-POCLOUD
   GHRSST Level 4 MUR 0.25deg Global Foundation Sea Surface Temperature Analysis (v4.2)...

3. C2586786218-POCLOUD
   GHRSST Level 4 OSTIA Global Historical Reprocessed Foundation Sea Surface Temperature Analysis produ...

4. C2089392421-NOAA_NCEI
   The Coral Reef Temperature Anomaly Database (CoRTAD) Version 6 - Global, 4 km Sea Surface Temperatur...

5. C1940475563-POCLOUD
   GHRSST Level 2P Global Sea Surface Skin Temperature from the Moderate Resolution Imaging Spectroradi...



---

## Test 2: LLM Reranker with Default Config

This configuration uses the new LLM reranker with default CMR-specific criteria:
- Variable Accuracy (25%)
- Resolution (20%)
- Processing Level (15%)
- Cross-Cutting Potential (10%)
- Ease of Use (10%)

### Configuration Details:
- `use_llm_reranker=True` - Enables the new LLM reranker
- `llm_reranker_model` - Specifies which model to use
- `llm_reranker_temperature=0.0` - Deterministic scoring
- Default criteria are defined in the CMR handler






In [22]:
print("\n" + "="*80)
print("TEST 2: LLM RERANKER WITH DEFAULT CONFIG")
print("="*80 + "\n")

# Configure CMR handler WITH LLM reranker (default config)
cmr_config_default = CMRHandlerConfig(
    use_llm_reranker=True,  # Use new LLM reranker
    llm_reranker_model="gpt-4o-mini",  # or "gpt-4o"
    llm_reranker_temperature=0.0,
    mcp_endpoint=CMR_ENDPOINT,  # IMPORTANT: Set the endpoint
    # custom_llm_reranker_config is None, so default CMR criteria are used
)

# Configure agent
config_default = DataSearchAgentConfig(
    cmr=cmr_config_default,
    topic_splitting_model=MODEL,
    scientific_decomposition_model=MODEL,
    repository_routing_model=MODEL,
    auto_save=True,
    output_subdir=str(output_path / "default_llm_reranker"),
    debug=DEBUG,
)

# Create agent
agent_default = DataSearchAgent(config=config_default, debug=DEBUG)

print("Configuration:")
print(f"  - use_llm_reranker: True")
print(f"  - llm_reranker_model: gpt-4o-mini")
print(f"  - llm_reranker_temperature: 0.0")
print(f"  - custom_llm_reranker_config: None (using default CMR criteria)")
print(f"  - mcp_endpoint: {CMR_ENDPOINT}")
print("\nDefault Criteria:")
print("  - Variable Accuracy (25%)")
print("  - Resolution (20%)")
print("  - Processing Level (15%)")
print("  - Cross-Cutting Potential (10%)")
print("  - Ease of Use (10%)")
print("\nRunning search...")


TEST 2: LLM RERANKER WITH DEFAULT CONFIG

Configuration:
  - use_llm_reranker: True
  - llm_reranker_model: gpt-4o-mini
  - llm_reranker_temperature: 0.0
  - custom_llm_reranker_config: None (using default CMR criteria)
  - mcp_endpoint: http://localhost:8080/mcp/cmr/mcp

Default Criteria:
  - Variable Accuracy (25%)
  - Resolution (20%)
  - Processing Level (15%)
  - Cross-Cutting Potential (10%)
  - Ease of Use (10%)

Running search...


In [ ]:
# Run the search
input_params_default = DataSearchAgentInputSchema(query=TEST_QUERY)
result_default = await agent_default.arun(input_params_default)

# Display results
print("\n" + "="*80)
print("RESULTS - DEFAULT LLM RERANKER")
print("="*80)
print(f"Search ID: {result_default.search_metadata.get('search_id')}")
print(f"Topics Processed: {result_default.search_metadata.get('topics_processed', 0)}")
print(f"Total CMR Results: {result_default.total_cmr_results}")
print(f"Total Filtered Results: {result_default.total_filtered_results}")
print(f"Duration: {result_default.search_metadata.get('duration_seconds', 0):.1f}s")

# Save results
output_file_default = output_path / "default_llm_reranker" / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
output_file_default.parent.mkdir(parents=True, exist_ok=True)
with open(output_file_default, "w") as f:
    json.dump(result_default.model_dump(), f, indent=2)

print(f"\nResults saved to: {output_file_default}")

In [24]:
# Display top 5 results with scoring details
print("\nTop 5 Collections (Default LLM Reranker):")
print("="*80)

if result_default.summary and result_default.summary.topics:
    for topic in result_default.summary.topics[:1]:  # Show first topic
        for decomp in topic.decompositions[:1]:  # Show first decomposition
            for idx, (concept_id, title) in enumerate(decomp.collections[:5], 1):
                print(f"{idx}. {concept_id}")
                print(f"   {title[:100]}...")
                print()


Top 5 Collections (Default LLM Reranker):
1. C2586786218-POCLOUD
   GHRSST Level 4 OSTIA Global Historical Reprocessed Foundation Sea Surface Temperature Analysis produ...

2. C1588876556-EUMETSAT
   SLSTR Sea Surface Temperatures (SST) in NRT - Sentinel-3...

3. C1588876559-EUMETSAT
   SLSTR Sea Surface Temperatures (SST) in NTC - Sentinel-3...

4. C1996881146-POCLOUD
   GHRSST Level 4 MUR Global Foundation Sea Surface Temperature Analysis (v4.1)...

5. C1940475563-POCLOUD
   GHRSST Level 2P Global Sea Surface Skin Temperature from the Moderate Resolution Imaging Spectroradi...



---

## Test 3: LLM Reranker with Custom Config

This configuration allows you to define your own scoring criteria, weights, and categories.

### How to Customize:

1. **Define Fields to Evaluate**: Specify which fields from the results to consider
2. **Create Scoring Criteria**: Define what aspects to evaluate (e.g., relevancy, quality)
3. **Set Weights**: Assign importance to each criterion (weights are auto-normalized)
4. **Define Categories**: For each criterion, define scoring categories with values

### Example Custom Configuration:

Below is an example that emphasizes relevancy and data quality over other factors.

In [25]:
print("\n" + "="*80)
print("TEST 3: LLM RERANKER WITH CUSTOM CONFIG")
print("="*80 + "\n")

# Define custom reranker configuration
custom_reranker_config = LLMRerankerToolConfig(
    model_name="gpt-4o-mini",  # or "gpt-4o" for better quality
    temperature=0.0,
    
    # Define which fields to extract from results for evaluation
    fields_to_evaluate={
        "title": "The title or name of the dataset (Entry Title in CMR)",
        "content": "Description or abstract of the dataset",
        "spatial_resolution": "Ground sampling distance - lower values (e.g., 30m) indicate higher resolution",
        "temporal_resolution": "Revisit time or frequency of data collection (e.g., daily, monthly)",
        "processing_level": "Data processing level (L0/1A=raw, L1/1B=calibrated, L2+=derived products)",
        "bounding_box": "Spatial extent of the dataset",
        "temporal": "Temporal coverage range of the dataset",
    },
    
    # Define custom scoring criteria
    scoring_criteria=[
        ScoringCriterion(
            name="Relevancy",
            description="How well does this dataset match the specific query requirements?",
            weight=0.5,  # 50% of total score
            scoring_categories=[
                ScoringCategory(
                    name="Highly Relevant",
                    description="Dataset directly addresses all aspects of the query",
                    value=3.0,
                ),
                ScoringCategory(
                    name="Partially Relevant",
                    description="Dataset addresses some aspects but misses key requirements",
                    value=2.0,
                ),
                ScoringCategory(
                    name="Not Relevant",
                    description="Dataset does not match query requirements",
                    value=0.0,
                ),
            ],
        ),
        ScoringCriterion(
            name="Data Quality",
            description="Is this a high-quality, well-processed dataset?",
            weight=0.3,  # 30% of total score
            scoring_categories=[
                ScoringCategory(
                    name="High Quality",
                    description="Level 2+ processing, well-documented, ready to use",
                    value=3.0,
                ),
                ScoringCategory(
                    name="Medium Quality",
                    description="Level 1 processing, requires some additional work",
                    value=2.0,
                ),
                ScoringCategory(
                    name="Low Quality",
                    description="Raw data or poorly documented",
                    value=1.0,
                ),
            ],
        ),
        ScoringCriterion(
            name="Usability",
            description="How easy is it to access and use this dataset?",
            weight=0.2,  # 20% of total score
            scoring_categories=[
                ScoringCategory(
                    name="Easy to Use",
                    description="Direct access, good documentation, standard formats",
                    value=3.0,
                ),
                ScoringCategory(
                    name="Moderate Effort",
                    description="Requires authentication or specialized tools",
                    value=2.0,
                ),
                ScoringCategory(
                    name="Difficult",
                    description="Hard to access or requires significant processing",
                    value=1.0,
                ),
            ],
        ),
    ],
)

# Configure CMR handler with custom LLM reranker config
cmr_config_custom = CMRHandlerConfig(
    use_llm_reranker=True,
    custom_llm_reranker_config=custom_reranker_config,  # Use custom config
    mcp_endpoint=CMR_ENDPOINT,  # IMPORTANT: Set the endpoint
)

# Configure agent
config_custom = DataSearchAgentConfig(
    cmr=cmr_config_custom,
    topic_splitting_model=MODEL,
    scientific_decomposition_model=MODEL,
    repository_routing_model=MODEL,
    auto_save=True,
    output_subdir=str(output_path / "custom_llm_reranker"),
    debug=DEBUG,
)

# Create agent
agent_custom = DataSearchAgent(config=config_custom, debug=DEBUG)

print("Configuration:")
print(f"  - use_llm_reranker: True")
print(f"  - custom_llm_reranker_config: Provided")
print(f"  - mcp_endpoint: {CMR_ENDPOINT}")
print("\nCustom Criteria:")
print("  - Relevancy (50%)")
print("  - Data Quality (30%)")
print("  - Usability (20%)")
print("\nRunning search...")


TEST 3: LLM RERANKER WITH CUSTOM CONFIG

Configuration:
  - use_llm_reranker: True
  - custom_llm_reranker_config: Provided
  - mcp_endpoint: http://localhost:8080/mcp/cmr/mcp

Custom Criteria:
  - Relevancy (50%)
  - Data Quality (30%)
  - Usability (20%)

Running search...


In [ ]:
# Run the search
input_params_custom = DataSearchAgentInputSchema(query=TEST_QUERY)
result_custom = await agent_custom.arun(input_params_custom)

# Display results
print("\n" + "="*80)
print("RESULTS - CUSTOM LLM RERANKER")
print("="*80)
print(f"Search ID: {result_custom.search_metadata.get('search_id')}")
print(f"Topics Processed: {result_custom.search_metadata.get('topics_processed', 0)}")
print(f"Total CMR Results: {result_custom.total_cmr_results}")
print(f"Total Filtered Results: {result_custom.total_filtered_results}")
print(f"Duration: {result_custom.search_metadata.get('duration_seconds', 0):.1f}s")

# Save results
output_file_custom = output_path / "custom_llm_reranker" / f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
output_file_custom.parent.mkdir(parents=True, exist_ok=True)
with open(output_file_custom, "w") as f:
    json.dump(result_custom.model_dump(), f, indent=2)

print(f"\nResults saved to: {output_file_custom}")

In [27]:
# Display top 5 results with scoring details
print("\nTop 5 Collections (Custom LLM Reranker):")
print("="*80)

if result_custom.summary and result_custom.summary.topics:
    for topic in result_custom.summary.topics[:1]:  
        for decomp in topic.decompositions[:1]:  
            for idx, (concept_id, title) in enumerate(decomp.collections[:5], 1):
                print(f"{idx}. {concept_id}")
                print(f"   {title[:100]}...")
                print()


Top 5 Collections (Custom LLM Reranker):
1. C2586786218-POCLOUD
   GHRSST Level 4 OSTIA Global Historical Reprocessed Foundation Sea Surface Temperature Analysis produ...

2. C1996881146-POCLOUD
   GHRSST Level 4 MUR Global Foundation Sea Surface Temperature Analysis (v4.1)...

3. C1940475563-POCLOUD
   GHRSST Level 2P Global Sea Surface Skin Temperature from the Moderate Resolution Imaging Spectroradi...

4. C2036880657-POCLOUD
   GHRSST Level 4 MUR 0.25deg Global Foundation Sea Surface Temperature Analysis (v4.2)...

5. C2805331435-POCLOUD
   GHRSST NOAA/STAR ACSPO v2.81 0.02 degree L3S Dataset from Afternoon LEO Satellites...



---

## Comparison Summary

Compare the results from all three configurations:

In [28]:
print("\n" + "="*80)
print("COMPARISON SUMMARY")
print("="*80)

print("\nPerformance Metrics:")
print("-" * 80)
print(f"{'Configuration':<30} {'Duration (s)':<15} {'Total Results':<15} {'Filtered':<15}")
print("-" * 80)

configs = [
    ("Legacy Reranker", result_legacy),
    ("Default LLM Reranker", result_default),
    ("Custom LLM Reranker", result_custom),
]

for name, result in configs:
    duration = result.search_metadata.get('duration_seconds', 0)
    total = result.total_cmr_results
    filtered = result.total_filtered_results
    print(f"{name:<30} {duration:<15.1f} {total:<15} {filtered:<15}")

print("\nTop 3 Collections Comparison:")
print("-" * 80)

for name, result in configs:
    print(f"\n{name}:")
    if result.summary and result.summary.topics:
        for topic in result.summary.topics[:1]:
            for decomp in topic.decompositions[:1]:
                for idx, (concept_id, title) in enumerate(decomp.collections[:3], 1):
                    print(f"  {idx}. {concept_id}: {title[:60]}...")


COMPARISON SUMMARY

Performance Metrics:
--------------------------------------------------------------------------------
Configuration                  Duration (s)    Total Results   Filtered       
--------------------------------------------------------------------------------
Legacy Reranker                145.2           32955           101            
Default LLM Reranker           299.9           34045           161            
Custom LLM Reranker            309.2           34363           221            

Top 3 Collections Comparison:
--------------------------------------------------------------------------------

Legacy Reranker:
  1. C1996881146-POCLOUD: GHRSST Level 4 MUR Global Foundation Sea Surface Temperature...
  2. C2036880657-POCLOUD: GHRSST Level 4 MUR 0.25deg Global Foundation Sea Surface Tem...
  3. C2586786218-POCLOUD: GHRSST Level 4 OSTIA Global Historical Reprocessed Foundatio...

Default LLM Reranker:
  1. C2586786218-POCLOUD: GHRSST Level 4 OSTIA Global His

In [ ]:
# get all collection IDs from each result set for comparison
legacy_ids = set()
default_ids = set()
custom_ids = set()

for topic in result_legacy.topics:
    for decomp in topic.decomposition_results:
        for collection in decomp.data_results:
            legacy_ids.add(collection.concept_id)

for topic in result_default.topics:
    for decomp in topic.decomposition_results:
        for collection in decomp.data_results:
            default_ids.add(collection.concept_id)


for topic in result_custom.topics:
    for decomp in topic.decomposition_results:
        for collection in decomp.data_results:
            custom_ids.add(collection.concept_id)

print("\nUnique Collections Found:")
print(f"  - Legacy Reranker: {len(legacy_ids)}")
print(f"  - Default LLM Reranker: {len(default_ids)}")
print(f"  - Custom LLM Reranker: {len(custom_ids)}")

AttributeError: 'str' object has no attribute 'concept_id'

In [30]:
print("\n All collection ids found in each configuration:")
print(f"  - Legacy Reranker: {sorted(legacy_ids)}")
print(f"  - Default LLM Reranker: {sorted(default_ids)}")   
print(f"  - Custom LLM Reranker: {sorted(custom_ids)}")


 All collection ids found in each configuration:
  - Legacy Reranker: ['C1940473819-POCLOUD', 'C1940475563-POCLOUD', 'C1996881146-POCLOUD', 'C2036880657-POCLOUD', 'C2089392421-NOAA_NCEI', 'C2089393309-NOAA_NCEI', 'C2213642059-GHRSSTCWIC', 'C2213645212-GHRSSTCWIC', 'C2270392799-POCLOUD', 'C2586786218-POCLOUD', 'C2698131226-JAXA', 'C2805331435-POCLOUD', 'C3160685780-OB_CLOUD', 'C3327360215-FEDEO', 'C3380709124-OB_CLOUD', 'C3380709133-OB_CLOUD', 'C3384236979-OB_CLOUD', 'C3384237428-OB_CLOUD', 'C3388381264-OB_CLOUD', 'C3388381565-OB_CLOUD', 'C3473420592-POCLOUD', 'C3478934705-POCLOUD', 'C3779578237-OB_CLOUD', 'C3779578265-OB_CLOUD', 'C3779578837-OB_CLOUD', 'C3779578885-OB_CLOUD']
  - Default LLM Reranker: ['C1570116979-OB_DAAC', 'C1570118532-OB_DAAC', 'C1588876556-EUMETSAT', 'C1588876559-EUMETSAT', 'C1940475563-POCLOUD', 'C1996881146-POCLOUD', 'C2036880657-POCLOUD', 'C2089392999-NOAA_NCEI', 'C2089393308-NOAA_NCEI', 'C2089393309-NOAA_NCEI', 'C2213642059-GHRSSTCWIC', 'C2213645212-GHRSSTCWIC

---

## Next Steps for SMEs

### 1. Test with Different Queries

Modify the `TEST_QUERY` variable in the Configuration section to test different types of queries:
- Simple queries (single variable, location, time)
- Complex queries (multiple variables, conditions)
- Domain-specific queries

### 2. Customize the Scoring Criteria

In Test 3, modify the `custom_reranker_config` to:
- Add new criteria relevant to your domain
- Adjust weights based on importance
- Define custom categories for each criterion
- Change which fields are evaluated

### 3. Compare Results

- Review the top results from each configuration
- Check the JSON output files for detailed scoring information
- Compare ranking orders and identify differences
- Validate results against your domain expertise

### 4. Provide Feedback

Document your observations:
- Which configuration produces the best results for your use case?
- Are there any critical datasets missing from the top results?
- Are there any irrelevant datasets ranked too highly?
- What criteria would improve ranking quality?

### 5. Advanced Customization

For more advanced use cases, you can:
- Modify the `fields_to_evaluate` dictionary to include additional metadata fields
- Create domain-specific scoring categories
- Adjust the number of results returned (`final_collection_count` in CMRHandlerConfig)
- Test different LLM models (gpt-4o vs gpt-4o-mini)

Default parameters of LLM Reranker 

### Default parameters
```json
{
  "model_name": "self.config.llm_reranker_model",
  "temperature": 0.0,
  "fields_to_evaluate": {
    "title": "Dataset name or entry title (CMR Entry Title)",
    "content": "Dataset description or abstract",
    "spatial_resolution": "Ground sampling distance (lower values like 30m indicate higher resolution)",
    "temporal_resolution": "Data collection frequency or revisit time (e.g., daily, monthly)",
    "processing_level": "Data processing stage (L0/1A=raw, L1/1B=calibrated/geolocated, L2+=derived with corrections)",
    "bounding_box": "Geographic extent of the dataset",
    "temporal": "Time range covered by the dataset"
  },
  "scoring_criteria": [
    {
      "name": "Variable Accuracy",
      "description": "Evaluates whether the dataset directly measures or derives the target variable",
      "weight": 0.25,
      "scoring_categories": [
        {
          "name": "Direct Measurement",
          "description": "Dataset directly measures the queried variable",
          "value": 3.0
        },
        {
          "name": "Indirectly Related",
          "description": "Dataset measures a parameter convertible to the desired variable",
          "value": 2.0
        },
        {
          "name": "Unrelated",
          "description": "Dataset measures unrelated phenomena",
          "value": 1.0
        }
      ]
    },
    {
      "name": "Resolution",
      "description": "Assesses spatial and temporal resolution alignment with the scientific question",
      "weight": 0.20,
      "scoring_categories": [
        {          "name": "Appropriate Spatial & Temporal Resolution",
          "description": "Dataset resolution matches the scientific question's scale",
          "value": 3.0
        },
        {
          "name": "Acceptable but Not Optimal",
          "description": "Resolution is usable but suboptimal",
          "value": 2.0
        },
        {
          "name": "Too Coarse or Too Infrequent",
          "description": "Resolution insufficient to capture the phenomenon",
          "value": 1.0
        }
      ]
    },
    {
      "name": "Processing Level",
      "description": "Indicates preprocessing applied to transform raw data into geophysical variables",
      "weight": 0.15,
      "scoring_categories": [
        {
          "name": "High Processing Level",
          "description": "Level 2+: calibrated, geolocated, atmospherically corrected or gridded",
          "value": 3.0
        },
        {
          "name": "Moderate Processing Level",
          "description": "Level 1/1B: calibrated but requires user-derived variables",
          "value": 2.0
        },
        {
          "name": "Raw",
          "description": "Level 0/1A: raw data needing extensive processing",
          "value": 1.0
        }
      ]
    },
    {
      "name": "Cross-Cutting Potential",
      "description": "Evaluates dataset combinability with others for multi-faceted queries",
      "weight": 0.10,
      "scoring_categories": [
        {       "name": "High Synergy",
          "description": "Dataset complements other instruments, enabling integrated analysis",
          "value": 3.0
        },
        {
          "name": "Moderate Synergy",
          "description": "Combinable but requires significant processing",
          "value": 2.0
        },
        {
          "name": "Limited Synergy",
          "description": "Unique or incompatible, minimal combined benefit",
          "value": 1.0
        }
      ]
    },
    {
      "name": "Ease of Use",
      "description": "Assesses simplicity of discovery, access, and use",
      "weight": 0.10,
      "scoring_categories": [
        {
          "name": "Easy Access and Well-Documented",
          "description": "Direct download with comprehensive documentation",
          "value": 3.0
        },
        {
          "name": "Moderate Usability",
          "description": "Requires authentication or specialized software",
          "value": 2.0
        },
        {
          "name": "Difficult Access/Poor Documentation",
          "description": "Requires special requests or has incomplete metadata",
          "value": 1.0
        }
      ]
    }
  ]
   
}
```